# 하수관로 결함 탐지 학습 (YOLOv8)
**사전 준비**: `sewer_yolo_dataset.zip`을 Google Drive 최상위(내 드라이브)에 업로드해둘 것.
**런타임**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택.
셀을 위에서부터 순서대로 실행.

In [ ]:
# ① Drive 마운트 + 데이터셋 압축 해제 (~10분)
from google.colab import drive
drive.mount('/content/drive')
!unzip -q /content/drive/MyDrive/sewer_yolo_dataset.zip -d /content/data
!ls /content/data/dataset/yolo

In [ ]:
# ② dataset.yaml 경로를 Colab 기준으로 수정
import re
p = "/content/data/dataset/yolo/dataset.yaml"
t = open(p, encoding="utf-8").read()
t = re.sub(r"path: .*", "path: /content/data/dataset/yolo", t)
open(p, "w", encoding="utf-8").write(t)
print(t)

In [ ]:
# ③ ultralytics 설치 + 학습 (T4 기준 약 2~4시간)
!pip install -q ultralytics
from ultralytics import YOLO
model = YOLO("yolov8s.pt")
results = model.train(data="/content/data/dataset/yolo/dataset.yaml",
                      epochs=50, imgsz=640, batch=-1, patience=10,
                      project="/content/runs", name="sewer")

In [ ]:
# ④ 클래스별 성능 (발표 슬라이드용 — 이 출력 캡처)
m = YOLO("/content/runs/sewer/weights/best.pt")
metrics = m.val(data="/content/data/dataset/yolo/dataset.yaml", split="test")
names = ["CL","CC","SD","BK","LP","JF","JD","DS","ETC"]
print(f"mAP50: {metrics.box.map50:.4f} | mAP50-95: {metrics.box.map:.4f}")
for i, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
    print(f"{names[int(i)]}: AP50 = {ap:.4f}")

In [ ]:
# ⑤ 결과물을 Drive에 백업 (가중치 + 학습 곡선)
!cp /content/runs/sewer/weights/best.pt /content/drive/MyDrive/sewer_best.pt
!cp /content/runs/sewer/results.png /content/drive/MyDrive/sewer_results.png
print("저장 완료: 내 드라이브에 sewer_best.pt, sewer_results.png")